In [1]:
import synapseclient
import pyarrow as pa
from pyarrow import fs
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pandas as pd
import re
from collections import defaultdict
import time
# import duckdb
# import rpy2
# import rpy2.robjects as robjects
# %load_ext rpy2.ipython

########
# Set up Access and download dataset
########
syn = synapseclient.Synapse()
syn.login()

ARCHIVE_VERSION = '2024-05-21'
COHORT = 'adults'
##just make sure that all of them are in one cohort as initially we are considering the dataset at cohort level
## since we have all data partitioned as ARCHIVE_VERSION/COHORT/DATASET_TYPE/PARTICIPANT_ID/part-x.parquet

########
#### Set up access and Get list of valid datasets
#### archived versions of the external parquet dataset (syn52506069)
########
## Set up Token access
sts_token = syn.get_sts_storage_token(entity='syn54128737', permission='read_only', output_format='json')

s3_external = fs.S3FileSystem(
    access_key=sts_token['accessKeyId'],
    secret_key=sts_token['secretAccessKey'],
    session_token=sts_token['sessionToken'],
    region='us-east-1'
)

## Get list of datasets in the S3 bucket
base_s3_uri_external = f"{sts_token['bucket']}/{sts_token['baseKey']}/{ARCHIVE_VERSION}/{COHORT}"
selector = fs.FileSelector(base_s3_uri_external, recursive=False)
parquet_datasets_external = s3_external.get_file_info(selector)

## Get all valid datasets
valid_paths = []
i = 0
for dataset in parquet_datasets_external:
    if re.search(r'recover-velsera-integration/main/archive/', dataset.path, re.IGNORECASE):
        i += 1
        print(f"{i}: {dataset.path}")
        valid_paths.append(dataset.path)

## Get dataset type (for eg., dataset_enrolledparticipants)
valid_paths_ext_df = pd.DataFrame(valid_paths, columns=['parquet_path_external'])
valid_paths_ext_df['datasetType'] = valid_paths_ext_df['parquet_path_external'].apply(lambda x: x.split('/')[5])

##############
## Read Fitbit Intraday in chunks.
## Each chunk contains all data for a select set of participants
##############
subset_paths_df = valid_paths_ext_df[valid_paths_ext_df['datasetType'] == 'dataset_fitbitintradaycombined'].reset_index()

participant_counts = defaultdict(int)

# get dataset path
path = subset_paths_df['parquet_path_external'][0]

# dataset at cohort level (eg. fitbitintradaycombined/adults/)
dataset_cohort = ds.dataset(path, filesystem=s3_external)

# Required list of participants 
## list of required participants; just make sure that all of them are in one cohort as initially we are considering
## the dataset at cohort level
# a few adult participants, some participant can have multiple parquet files associated with them
participant_list = ['RA11003-00024', 'RA11001-00250', 'RA11004-00028']
file_list = [] # initiate list of files that represent the subset of data corresponding to participant_list


# Find the required parquet paths from dataset.files, i,e just the participants we want
for file_parquet in dataset_cohort.files:
    participant_id = file_parquet.split('/')[6]
    if participant_id in participant_list:
        # print(participant_id)
        file_list.append(file_parquet)

# Create a dataset from file_list, this would correspond to only the required participants
dataset = ds.dataset(file_list, filesystem=s3_external, format='parquet')

Welcome, yj167000!

1: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_enrolledparticipants
2: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_enrolledparticipants_customfields_symptoms
3: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_enrolledparticipants_customfields_treatments
4: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitactivitylogs
5: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitdailydata
6: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitdevices
7: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitecg
8: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitecg_waveformsamples
9: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitintradaycombined
10: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitrestingheartrates
11: recover-velsera-integration/main/a

In [13]:
len(dataset_cohort.files)

3602

In [2]:
intra_comb = dataset.to_table().to_pandas()
intra_comb['Type'].unique()

array(['activities-steps', 'activities-floors', 'activities-elevation',
       'activities-distance', 'activities-calories', 'activities-heart',
       'hrv', 'spo2', 'br'], dtype=object)

In [6]:
intra_hr = [intra_comb['Type']=='activities-heart']

In [8]:
intra_comb

,ParticipantID,ParticipantIdentifier,Type,DateTime,Value,Level,Mets,InsertedDate,export_start_date,export_end_date,Rmssd,Coverage,Hf,Lf,DeepSleepSummaryBreathRate,RemSleepSummaryBreathRate,FullSleepSummaryBreathRate,LightSleepSummaryBreathRate,cohort
0,8d82d777-c87c-4469-8712-d5a532d396a3,RA11001-00250,activities-steps,2024-03-02T04:47:00,0,None,None,2024-03-04T04:19:41Z,2024-03-04T00:00:00,2024-03-05T00:00:00,None,None,None,None,None,None,None,None,adults_v1
1,8d82d777-c87c-4469-8712-d5a532d396a3,RA11001-00250,activities-steps,2024-03-02T07:14:00,0,None,None,2024-03-04T04:19:41Z,2024-03-04T00:00:00,2024-03-05T00:00:00,None,None,None,None,None,None,None,None,adults_v1
2,8d82d777-c87c-4469-8712-d5a532d396a3,RA11001-00250,activities-steps,2024-03-02T07:33:00,0,None,None,2024-03-04T04:19:41Z,2024-03-04T00:00:00,2024-03-05T00:00:00,None,None,None,None,None,None,None,None,adults_v1
3,8d82d777-c87c-4469-8712-d5a532d396a3,RA11001-00250,activities-steps,2024-03-02T07:59:00,0,None,None,2024-03-04T04:19:41Z,2024-03-04T00:00:00,2024-03-05T00:00:00,None,None,None,None,None,None,None,None,adults_v1
4,8d82d777-c87c-4469-8712-d5a532d396a3,RA11001-00250,activities-steps,2024-03-02T08:04:00,0,None,None,2024-03-04T04:19:41Z,2024-03-04T00:00:00,2024-03-05T00:00:00,None,None,None,None,None,None,None,None,adults_v1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9238277,a30e91fc-572e-4a12-88dc-8ecae45e8892,RA11004-00028,activities-steps,2024-05-18T00:29:00,0,None,None,2024-05-18T13:33:34Z,2024-05-18T00:00:00,2024-05-19T00:00:00,None,None,None,None,None,None,None,None,adults_v1
9238278,a30e91fc-572e-4a12-88dc-8ecae45e8892,RA11004-00028,activities-steps,2024-05-18T02:53:00,0,None,None,2024-05-18T13:33:34Z,2024-05-18T00:00:00,2024-05-19T00:00:00,None,None,None,None,None,None,None,None,adults_v1
9238279,a30e91fc-572e-4a12-88dc-8ecae45e8892,RA11004-00028,activities-steps,2024-05-18T03:02:00,0,None,None,2024-05-18T13:33:34Z,2024-05-18T00:00:00,2024-05-19T00:00:00,None,None,None,None,None,None,None,None,adults_v1
9238280,a30e91fc-572e-4a12-88dc-8ecae45e8892,RA11004-00028,activities-steps,2024-05-18T06:38:00,0,None,None,2024-05-18T13:33:34Z,2024-05-18T00:00:00,2024-05-19T00:00:00,None,None,None,None,None,None,None,None,adults_v1
